In [1]:
import numpy as np
import torch
import torch.nn.functional as F
import time, os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

# ==========================================
# HYPERPARAMETERS
# ==========================================
class Phase1Config:
    N_SCHEMES = 20  # Number of random seeds to attempt
    EPOCHS = 15000  # Epochs to find a rank-23 solution
    LR = 0.005
    NOISE_INIT = 0.1
    PHASES = [
        (0, 3000, 0.10, 0.005, 0.0),
        (3000, 8000, 0.04, 0.002, 1e-6),
        (8000, 12000, 0.01, 0.001, 1e-5),
        (12000, 15000, 0.0, 0.0005, 1e-5),
    ]
    VERBOSE_INTERVAL = 3000
    CONVERGE_THRESHOLD = 1e-6  # Anything below this is an exact rank-23 root
    PLATEAU_PATIENCE = 1000
    PLATEAU_BURST = 0.03

class Phase2Config:
    EPOCHS_PER_DROP = 15000
    PHASES = [
        (0, 3000, 0.12, 0.005, 0.0),
        (3000, 8000, 0.06, 0.002, 1e-5),
        (8000, 13000, 0.02, 0.001, 1e-4),
        (13000, 15000, 0.0, 0.0005, 0.0),
    ]
    NOISE_INIT = 0.05
    PLATEAU_PATIENCE = 800
    PLATEAU_BURST = 0.04
    VERBOSE_INTERVAL = 3000
    BREAKTHROUGH_THRESHOLD = 5e-4 # Our target to beat the Laderman 7e-3 floor

cfg1 = Phase1Config()
cfg2 = Phase2Config()

# ==========================================
# TARGET TENSOR & UTILS
# ==========================================
def build_target():
    T = torch.zeros((9,9,9), dtype=torch.float32, device=DEVICE)
    for i in range(3):
        for j in range(3):
            for k in range(3):
                T[i*3+j, j*3+k, i*3+k] = 1.0
    return T

T_target = build_target()

def random_orthogonal(dim):
    H = torch.randn(dim, dim, device=DEVICE)
    Q, R = torch.linalg.qr(H)
    return Q * torch.sign(torch.diag(R)).unsqueeze(0)

def apply_basis(T, P, Q, R):
    T1 = torch.einsum("ia,abc->ibc", P, T)
    T2 = torch.einsum("jb,ibc->ijc", Q, T1)
    T3 = torch.einsum("kc,ijc->ijk", R, T2)
    return T3

def get_params(epoch, phases):
    for s,e,n,lr,l1 in phases:
        if s <= epoch < e:
            frac = (epoch-s)/(e-s) if e>s else 0.0
            return n*(1-0.5*frac), lr, l1
    return 0.0, 1e-4, phases[-1][4]

# ==========================================
# PHASE 1: DISCOVER FROM CHAOS
# ==========================================
def discover_scheme(tid):
    start_t = time.time()
    
    # Random orthogonal basis to rotate the target
    P, Q, R = random_orthogonal(9), random_orthogonal(9), random_orthogonal(9)
    T_trans = apply_basis(T_target, P, Q, R)
    
    # PURE CHAOS INITIALIZATION
    U = torch.randn(9, 23, device=DEVICE) * cfg1.NOISE_INIT
    V = torch.randn(9, 23, device=DEVICE) * cfg1.NOISE_INIT
    W = torch.randn(9, 23, device=DEVICE) * cfg1.NOISE_INIT
    U.requires_grad_(True); V.requires_grad_(True); W.requires_grad_(True)
    
    opt = torch.optim.Adam([U, V, W], lr=cfg1.LR)
    best_loss = float("inf"); best_state = None; patience = 0
    
    for ep in range(cfg1.EPOCHS):
        nmag, lr_curr, l1s = get_params(ep, cfg1.PHASES)
        for g in opt.param_groups: g["lr"] = lr_curr
        opt.zero_grad()
        
        recon = torch.einsum("ir,jr,kr->ijk", U, V, W)
        mse = F.mse_loss(recon, T_trans)
        reg = l1s * (U.abs().mean() + V.abs().mean() + W.abs().mean())
        (mse + reg).backward()
        opt.step()
        
        if nmag > 0:
            with torch.no_grad():
                U.add_(torch.randn_like(U) * nmag)
                V.add_(torch.randn_like(V) * nmag)
                W.add_(torch.randn_like(W) * nmag)
        
        cv = mse.item()
        if cv < best_loss:
            best_loss = cv
            best_state = (U.detach().clone(), V.detach().clone(), W.detach().clone(), P, Q, R)
            patience = 0
        else:
            patience += 1
            
        if patience > cfg1.PLATEAU_PATIENCE and ep > cfg1.PHASES[1][0]:
            with torch.no_grad():
                U.add_(torch.randn_like(U) * cfg1.PLATEAU_BURST)
                V.add_(torch.randn_like(V) * cfg1.PLATEAU_BURST)
                W.add_(torch.randn_like(W) * cfg1.PLATEAU_BURST)
            patience = 0
            
        if ep % cfg1.VERBOSE_INTERVAL == 0:
            print(f"  [P1] S{tid:2d} Ep{ep:5d} MSE:{cv:.2e}")
            
        if cv < cfg1.CONVERGE_THRESHOLD:
            print(f"  [P1] S{tid:2d}: EXACT RANK-23 ROOT FOUND at ep {ep}!")
            break
            
    U_b, V_b, W_b, Pb, Qb, Rb = best_state
    with torch.no_grad():
        U_o = Pb.T @ U_b; V_o = Qb.T @ V_b; W_o = Rb.T @ W_b
        mse_o = F.mse_loss(torch.einsum("ir,jr,kr->ijk", U_o, V_o, W_o), T_target).item()
        
    return {"tid": tid, "mse": mse_o, "U": U_o.cpu(), "V": V_o.cpu(), "W": W_o.cpu(), "t": time.time()-start_t}

# ==========================================
# PHASE 2: THE DROP TEST
# ==========================================
def test_rank22(scheme_id, U_full, V_full, W_full, col_drop):
    start_t = time.time()
    keep = [i for i in range(23) if i != col_drop]
    
    U_init = U_full[:, keep].float().clone().to(DEVICE)
    V_init = V_full[:, keep].float().clone().to(DEVICE)
    W_init = W_full[:, keep].float().clone().to(DEVICE)
    
    U = U_init + cfg2.NOISE_INIT * torch.randn(9, 22, device=DEVICE)
    V = V_init + cfg2.NOISE_INIT * torch.randn(9, 22, device=DEVICE)
    W = W_init + cfg2.NOISE_INIT * torch.randn(9, 22, device=DEVICE)
    U.requires_grad_(True); V.requires_grad_(True); W.requires_grad_(True)
    
    opt = torch.optim.Adam([U, V, W], lr=0.005)
    best_loss = float("inf"); patience = 0
    
    for ep in range(cfg2.EPOCHS_PER_DROP):
        nmag, lr_curr, l1s = get_params(ep, cfg2.PHASES)
        for g in opt.param_groups: g["lr"] = lr_curr
        opt.zero_grad()
        
        mse = F.mse_loss(torch.einsum("ir,jr,kr->ijk", U, V, W), T_target)
        reg = l1s * (U.abs().mean() + V.abs().mean() + W.abs().mean())
        (mse + reg).backward()
        opt.step()
        
        if nmag > 0:
            with torch.no_grad():
                U.add_(torch.randn_like(U) * nmag)
                V.add_(torch.randn_like(V) * nmag)
                W.add_(torch.randn_like(W) * nmag)
                
        cv = mse.item()
        if cv < best_loss:
            best_loss = cv
            patience = 0
            if cv < 1e-6:
                print(f"  >>> EXACT CONTINUOUS RANK-22! Scheme {scheme_id} Col {col_drop}")
                break
        else:
            patience += 1
            
        if patience > cfg2.PLATEAU_PATIENCE and ep > cfg2.PHASES[1][0]:
            with torch.no_grad():
                U.add_(torch.randn_like(U) * cfg2.PLATEAU_BURST)
                V.add_(torch.randn_like(V) * cfg2.PLATEAU_BURST)
                W.add_(torch.randn_like(W) * cfg2.PLATEAU_BURST)
            patience = 0
            
    return {"scheme": scheme_id, "col": col_drop, "mse": best_loss, "t": time.time()-start_t}

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    print(f"\n{'='*60}\nPHASE 1: HUNTING NEW RANK-23 ROOTS\n{'='*60}")
    schemes = []
    
    for s in range(cfg1.N_SCHEMES):
        print(f"\n--- Root Hunt {s+1}/{cfg1.N_SCHEMES} ---")
        r = discover_scheme(s + 1)
        print(f"  >> Root {s+1} Final MSE: {r['mse']:.2e} | {r['t']:.0f}s")
        schemes.append(r)
        
    good_schemes = [s for s in schemes if s["mse"] < 1e-4]
    print(f"\nSuccessfully locked {len(good_schemes)} valid Rank-23 roots.")
    
    # Try to verify distance from Laderman (if file exists)
    try:
        d = np.load("factorizations_r.npz", allow_pickle=True)["3,3,3"].item()
        U_ref = torch.tensor(np.array(d["u"], dtype=np.float32))
        print("\nDistance from Laderman Anchor (Proving Novelty):")
        for s in good_schemes:
            diff = torch.norm(s["U"] - U_ref).item()
            print(f"  Scheme {s['tid']}: Frobenius Diff = {diff:.2f}")
    except:
        print("\n(Laderman reference file not found, skipping distance metric)")

    # Execute Phase 2
    MAX_SCHEMES_TO_TEST = 3
    candidates = sorted(good_schemes, key=lambda s: s["mse"])[:MAX_SCHEMES_TO_TEST]
    
    if len(candidates) > 0:
        print(f"\n{'='*60}\nPHASE 2: THE DROP TEST (Bypassing the Anchor)\n{'='*60}")
        breakthroughs = []
        
        for scheme in candidates:
            sid = scheme["tid"]
            print(f"\n--- Testing Scheme {sid} (Base MSE={scheme['mse']:.2e}) ---")
            
            for col in range(23):
                r = test_rank22(sid, scheme["U"], scheme["V"], scheme["W"], col)
                tag = ""
                if r["mse"] < cfg2.BREAKTHROUGH_THRESHOLD:
                    tag = " *** BREAKTHROUGH: SHATTERED LADERMAN FLOOR! ***"
                    breakthroughs.append(r)
                elif r["mse"] < 0.007:
                    tag = " (Below Laderman barrier)"
                    
                print(f"  Col {col:2d}: MSE={r['mse']:.2e} | {r['t']:.0f}s{tag}")
                
        if breakthroughs:
            print(f"\n!!! FOUND {len(breakthroughs)} BREAKTHROUGHS ALGEBRAICALLY CAPABLE OF RANK-22 !!!")
        else:
            print("\nNo continuous breakthroughs found. The search space requires broader exploration.")
    else:
        print("\nPhase 1 failed to converge any stable roots. Consider increasing cfg1.NOISE_INIT or EPOCHS.")

Device: cuda | PyTorch: 2.4.1+cu121

PHASE 1: HUNTING NEW RANK-23 ROOTS

--- Root Hunt 1/20 ---
  [P1] S 1 Ep    0 MSE:3.71e-02
  [P1] S 1 Ep 3000 MSE:3.89e+00
  [P1] S 1 Ep 6000 MSE:1.10e+00
  [P1] S 1 Ep 9000 MSE:9.96e-02
  [P1] S 1 Ep12000 MSE:3.01e-02
  >> Root 1 Final MSE: 1.45e-02 | 69s

--- Root Hunt 2/20 ---
  [P1] S 2 Ep    0 MSE:3.71e-02
  [P1] S 2 Ep 3000 MSE:3.05e+00
  [P1] S 2 Ep 6000 MSE:1.34e+00
  [P1] S 2 Ep 9000 MSE:1.27e-01
  [P1] S 2 Ep12000 MSE:2.97e-02
  >> Root 2 Final MSE: 1.31e-02 | 65s

--- Root Hunt 3/20 ---
  [P1] S 3 Ep    0 MSE:3.72e-02
  [P1] S 3 Ep 3000 MSE:2.90e+00
  [P1] S 3 Ep 6000 MSE:1.43e+00
  [P1] S 3 Ep 9000 MSE:1.02e-01
  [P1] S 3 Ep12000 MSE:3.00e-02
  >> Root 3 Final MSE: 1.57e-02 | 70s

--- Root Hunt 4/20 ---
  [P1] S 4 Ep    0 MSE:3.71e-02
  [P1] S 4 Ep 3000 MSE:2.86e+00
  [P1] S 4 Ep 6000 MSE:1.10e+00
  [P1] S 4 Ep 9000 MSE:1.02e-01
  [P1] S 4 Ep12000 MSE:2.97e-02
  >> Root 4 Final MSE: 1.46e-02 | 66s

--- Root Hunt 5/20 ---
  [P1] S 5 Ep   